In [1]:
# ============================================================
# 01_build_dataset.ipynb
# Single-receiver dataset builder for:
#   pretrain/source-train on A
#   fine-tune on B
#   joint retrain on A+B
#   evaluate unseen locations on both A and B
# ============================================================

from pathlib import Path
import json
import random
import re
import numpy as np
import pandas as pd
from scipy.io import loadmat

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path("/home/tonyliao/Location")
WINDOW_ROOT = DATA_ROOT
OUT_DIR = DATA_ROOT / "dataset_build_hybrid"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TARGET_EVAL_UNSEEN_LOCATIONS = {
    "Center",
    "Corner",
    "Empty_Pred",
    "LeftDown_Far",
    "LeftUp_Near",
    "LeftUp_Pred",
    "Wall",
}

A_VAL_RATIO = 0.2
B_VAL_RATIO = 0.2
KEEP_EMPTY = True

# -----------------------------
# Helpers
# -----------------------------
def normalize_label(label: str) -> str:
    label = str(label).strip()
    label = re.sub(r"\s+", "_", label)
    return label

def is_empty_label(label: str) -> bool:
    return normalize_label(label).lower() in {
        "empty", "empty_pred", "empty_unlab", "none", "no_person"
    }

def safe_scalar(x, default=None):
    if x is None:
        return default
    if isinstance(x, np.ndarray):
        if x.size == 0:
            return default
        if x.shape == ():
            return x.item()
        if x.size == 1:
            return x.reshape(-1)[0].item()
    return x

def try_load_meta(path: Path) -> dict:
    try:
        raw = loadmat(path, squeeze_me=True, struct_as_record=False)
        meta = raw.get("meta", None)
        if meta is None:
            return {}

        def get_field(name, default=None):
            if hasattr(meta, name):
                return safe_scalar(getattr(meta, name), default)
            return default

        return {
            "session_id": str(get_field("session_id", "")),
            "source_file": str(get_field("source_file", "")),
            "window_index": int(get_field("window_index", -1)),
            "is_flagged_bad": bool(get_field("is_flagged_bad", False)),
            "CF_MHz": float(get_field("CF_MHz", np.nan)),
            "CBW_MHz": float(get_field("CBW_MHz", np.nan)),
            "A_rx": int(get_field("A_rx", -1)),
            "S_kept": int(get_field("S_kept", -1)),
            "amp_mean": float(get_field("amp_mean", np.nan)),
            "amp_std": float(get_field("amp_std", np.nan)),
            "phase_std": float(get_field("phase_std", np.nan)),
        }
    except Exception:
        return {}

def infer_from_path(amp_path: Path):
    # expected:
    # /home/tonyliao/Location/A/Center/amp_window_00001.npy
    receiver_domain = amp_path.parent.parent.name
    anchor_label = normalize_label(amp_path.parent.name)
    return receiver_domain, anchor_label

def session_key_from_path(amp_path: Path):
    # with current layout, one location folder acts like one session bucket
    receiver_domain = amp_path.parent.parent.name
    anchor_label = normalize_label(amp_path.parent.name)
    return f"{receiver_domain}__{anchor_label}"

def scan_windows(root: Path) -> pd.DataFrame:
    rows = []

    amp_files = sorted(root.rglob("amp_window_*.npy"))
    print("Found amp files:", len(amp_files))

    for amp_path in amp_files:
        folder = amp_path.parent
        m = re.search(r"amp_window_(\d+)\.npy$", amp_path.name)
        if not m:
            continue
        idx = int(m.group(1))

        meta_path = folder / f"meta_{idx:05d}.mat"
        pha_path = folder / f"pha_window_{idx:05d}.npy"

        receiver_domain, anchor_label = infer_from_path(amp_path)
        meta = try_load_meta(meta_path) if meta_path.exists() else {}

        row = {
            "amp_path": str(amp_path),
            "pha_path": str(pha_path) if pha_path.exists() else "",
            "meta_path": str(meta_path) if meta_path.exists() else "",
            "receiver_domain": receiver_domain,
            "anchor_label": anchor_label,
            "session_id": "",  # do not trust MATLAB string object here
            "source_file": str(meta.get("source_file", "")),
            "window_index": int(meta.get("window_index", idx)),
            "is_flagged_bad": bool(meta.get("is_flagged_bad", False)),
            "CF_MHz": float(meta.get("CF_MHz", np.nan)),
            "CBW_MHz": float(meta.get("CBW_MHz", np.nan)),
            "A_rx": int(meta.get("A_rx", -1)),
            "S_kept": int(meta.get("S_kept", -1)),
            "amp_mean": float(meta.get("amp_mean", np.nan)),
            "amp_std": float(meta.get("amp_std", np.nan)),
            "phase_std": float(meta.get("phase_std", np.nan)),
        }

        row["presence"] = 0 if is_empty_label(row["anchor_label"]) else 1
        row["session_key"] = session_key_from_path(amp_path)
        rows.append(row)

    return pd.DataFrame(rows)

def split_within_label(df_label: pd.DataFrame, val_ratio=0.2, seed=42):
    idx = np.arange(len(df_label))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    if len(idx) <= 1:
        return df_label.copy(), df_label.iloc[0:0].copy()

    n_val = max(1, int(round(len(df_label) * val_ratio)))
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    return (
        df_label.iloc[train_idx].copy().reset_index(drop=True),
        df_label.iloc[val_idx].copy().reset_index(drop=True),
    )

def df_to_npz(df: pd.DataFrame, path: Path, label_map=None):
    obj = {}
    for col in df.columns:
        vals = df[col].tolist()
        if len(vals) == 0:
            obj[col] = np.array([], dtype=object)
            continue

        first = vals[0]
        if isinstance(first, (int, np.integer)):
            obj[col] = np.asarray(vals, dtype=np.int64)
        elif isinstance(first, (float, np.floating)):
            obj[col] = np.asarray(vals, dtype=np.float32)
        elif isinstance(first, (bool, np.bool_)):
            obj[col] = np.asarray(vals, dtype=bool)
        else:
            obj[col] = np.asarray(vals, dtype=object)

    if label_map is not None and "anchor_label" in df.columns:
        unknown_labels = sorted(set(map(str, df["anchor_label"].tolist())) - set(label_map.keys()))
        if unknown_labels:
            raise ValueError(f"These labels are missing from label_map: {unknown_labels}")
        label_ids = [label_map[str(x)] for x in df["anchor_label"].tolist()]
        obj["label_id"] = np.asarray(label_ids, dtype=np.int64)

    np.savez(path, **obj)

def assert_no_overlap(name1, df1, name2, df2, key="session_key"):
    s1 = set(df1[key].astype(str).tolist())
    s2 = set(df2[key].astype(str).tolist())
    overlap = s1 & s2
    if overlap:
        raise AssertionError(
            f"Overlap between {name1} and {name2}: {sorted(list(overlap))[:10]}"
        )

def assert_no_file_overlap(name1, df1, name2, df2, key="amp_path"):
    s1 = set(df1[key].astype(str).tolist())
    s2 = set(df2[key].astype(str).tolist())
    overlap = s1 & s2
    if overlap:
        raise AssertionError(
            f"Exact file overlap between {name1} and {name2}: {sorted(list(overlap))[:10]}"
        )

# -----------------------------
# Scan exported windows
# -----------------------------
df = scan_windows(WINDOW_ROOT)

print("Total windows:", len(df))
if len(df) == 0:
    raise ValueError("No windows found. Check path or filename pattern.")

print(df["receiver_domain"].value_counts(dropna=False))
print(df["anchor_label"].value_counts(dropna=False).head(20))

if not KEEP_EMPTY:
    df = df[df["presence"] == 1].copy()

df = df[df["receiver_domain"].isin(["A", "B"])].copy().reset_index(drop=True)

manifest_csv = OUT_DIR / "all_windows_manifest.csv"
df.to_csv(manifest_csv, index=False)
print("Saved manifest:", manifest_csv)

# -----------------------------
# Unseen-location policy
# Hold out unseen labels from BOTH A and B
# -----------------------------
target_unseen_loc_set = set(map(normalize_label, TARGET_EVAL_UNSEEN_LOCATIONS))

# known-label dataframe used to build supervised label vocabulary
known_label_df = df[
    ~df["anchor_label"].isin(target_unseen_loc_set)
].copy()

labels = sorted(known_label_df["anchor_label"].astype(str).unique().tolist())
label_map = {lab: i for i, lab in enumerate(labels)}
inv_label_map = {i: lab for lab, i in label_map.items()}

with open(OUT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_map": label_map,
            "inv_label_map": {str(k): v for k, v in inv_label_map.items()},
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("num_labels:", len(label_map))
print(label_map)

# -----------------------------
# Split A
# Known labels -> A_train / A_val
# Unseen labels -> A_eval_unseen_locations
# -----------------------------
df_A = df[df["receiver_domain"] == "A"].copy().reset_index(drop=True)

A_eval_unseen_locations = df_A[
    df_A["anchor_label"].isin(target_unseen_loc_set)
].copy().reset_index(drop=True)

A_known = df_A[
    ~df_A["anchor_label"].isin(target_unseen_loc_set)
].copy().reset_index(drop=True)

A_train_parts = []
A_val_parts = []

for label in sorted(A_known["anchor_label"].unique()):
    sub = A_known[A_known["anchor_label"] == label].copy().reset_index(drop=True)
    tr, va = split_within_label(sub, val_ratio=A_VAL_RATIO, seed=SEED)
    A_train_parts.append(tr)
    A_val_parts.append(va)

A_train = pd.concat(A_train_parts, axis=0).reset_index(drop=True)
A_val   = pd.concat(A_val_parts, axis=0).reset_index(drop=True)

print("\nA split label counts:")
print("A_train:")
print(A_train["anchor_label"].value_counts().sort_index())
print("A_val:")
print(A_val["anchor_label"].value_counts().sort_index())
print("A_eval_unseen_locations:")
print(A_eval_unseen_locations["anchor_label"].value_counts().sort_index())

# -----------------------------
# Split B
# Known labels -> B_finetune / B_val
# Unseen labels -> B_eval_unseen_locations
# -----------------------------
df_B = df[df["receiver_domain"] == "B"].copy().reset_index(drop=True)

B_eval_unseen_locations = df_B[
    df_B["anchor_label"].isin(target_unseen_loc_set)
].copy().reset_index(drop=True)

B_known = df_B[
    ~df_B["anchor_label"].isin(target_unseen_loc_set)
].copy().reset_index(drop=True)

B_finetune_parts = []
B_val_parts = []

for label in sorted(B_known["anchor_label"].unique()):
    sub = B_known[B_known["anchor_label"] == label].copy().reset_index(drop=True)
    tr, va = split_within_label(sub, val_ratio=B_VAL_RATIO, seed=SEED)
    B_finetune_parts.append(tr)
    B_val_parts.append(va)

B_finetune = pd.concat(B_finetune_parts, axis=0).reset_index(drop=True)
B_val      = pd.concat(B_val_parts, axis=0).reset_index(drop=True)

print("\nB split label counts:")
print("B_finetune:")
print(B_finetune["anchor_label"].value_counts().sort_index())
print("B_val:")
print(B_val["anchor_label"].value_counts().sort_index())
print("B_eval_unseen_locations:")
print(B_eval_unseen_locations["anchor_label"].value_counts().sort_index())

# -----------------------------
# Combined eval set
# -----------------------------
target_eval_unseen_locations_all = pd.concat(
    [A_eval_unseen_locations.copy(), B_eval_unseen_locations.copy()],
    axis=0
).reset_index(drop=True)

print("\nCombined unseen eval counts:")
print(target_eval_unseen_locations_all["receiver_domain"].value_counts().sort_index())
print(target_eval_unseen_locations_all["anchor_label"].value_counts().sort_index())

# -----------------------------
# Joint-retrain splits
# Stage 04b uses A_train + B_finetune for training
# and A_val + B_val for validation
# -----------------------------
AB_joint_retrain = pd.concat(
    [A_train.copy(), B_finetune.copy()],
    axis=0
).reset_index(drop=True)

AB_joint_val = pd.concat(
    [A_val.copy(), B_val.copy()],
    axis=0
).reset_index(drop=True)

print("\nAB joint retrain counts:")
print("AB_joint_retrain receiver counts:")
print(AB_joint_retrain["receiver_domain"].value_counts().sort_index())
print("AB_joint_val receiver counts:")
print(AB_joint_val["receiver_domain"].value_counts().sort_index())

# -----------------------------
# Safety checks
# -----------------------------
# within-known-label splits: exact file overlap not allowed
assert_no_file_overlap("A_train", A_train, "A_val", A_val)
assert_no_file_overlap("B_finetune", B_finetune, "B_val", B_val)

# unseen eval must remain separate from known splits
assert_no_overlap(
    "A_train", A_train,
    "A_eval_unseen_locations", A_eval_unseen_locations,
    key="session_key"
)
assert_no_overlap(
    "A_val", A_val,
    "A_eval_unseen_locations", A_eval_unseen_locations,
    key="session_key"
)
assert_no_overlap(
    "B_finetune", B_finetune,
    "B_eval_unseen_locations", B_eval_unseen_locations,
    key="session_key"
)
assert_no_overlap(
    "B_val", B_val,
    "B_eval_unseen_locations", B_eval_unseen_locations,
    key="session_key"
)

loc_overlap_a_tr = set(A_train["anchor_label"].tolist()) & set(A_eval_unseen_locations["anchor_label"].tolist())
if loc_overlap_a_tr:
    raise AssertionError(
        f"Location leakage between A_train and A_eval_unseen_locations: {sorted(list(loc_overlap_a_tr))}"
    )

loc_overlap_a_val = set(A_val["anchor_label"].tolist()) & set(A_eval_unseen_locations["anchor_label"].tolist())
if loc_overlap_a_val:
    raise AssertionError(
        f"Location leakage between A_val and A_eval_unseen_locations: {sorted(list(loc_overlap_a_val))}"
    )

loc_overlap_b_ft = set(B_finetune["anchor_label"].tolist()) & set(B_eval_unseen_locations["anchor_label"].tolist())
if loc_overlap_b_ft:
    raise AssertionError(
        f"Location leakage between B_finetune and B_eval_unseen_locations: {sorted(list(loc_overlap_b_ft))}"
    )

loc_overlap_b_val = set(B_val["anchor_label"].tolist()) & set(B_eval_unseen_locations["anchor_label"].tolist())
if loc_overlap_b_val:
    raise AssertionError(
        f"Location leakage between B_val and B_eval_unseen_locations: {sorted(list(loc_overlap_b_val))}"
    )

# joint retrain must not contain unseen eval files
assert_no_file_overlap(
    "AB_joint_retrain", AB_joint_retrain,
    "target_eval_unseen_locations_all", target_eval_unseen_locations_all
)
assert_no_file_overlap(
    "AB_joint_val", AB_joint_val,
    "target_eval_unseen_locations_all", target_eval_unseen_locations_all
)

# -----------------------------
# Save splits
# -----------------------------
splits = {
    "A_train": A_train.reset_index(drop=True),
    "A_val": A_val.reset_index(drop=True),
    "B_finetune": B_finetune.reset_index(drop=True),
    "B_val": B_val.reset_index(drop=True),
    "A_eval_unseen_locations": A_eval_unseen_locations.reset_index(drop=True),
    "B_eval_unseen_locations": B_eval_unseen_locations.reset_index(drop=True),
    "target_eval_unseen_locations_all": target_eval_unseen_locations_all.reset_index(drop=True),
    "AB_joint_retrain": AB_joint_retrain.reset_index(drop=True),
    "AB_joint_val": AB_joint_val.reset_index(drop=True),
}

for name, sdf in splits.items():
    sdf.to_csv(OUT_DIR / f"{name}.csv", index=False)

    # only known-label supervised splits get label_id
    if name in {"A_train", "A_val", "B_finetune", "B_val", "AB_joint_retrain", "AB_joint_val"}:
        df_to_npz(sdf, OUT_DIR / f"{name}.npz", label_map=label_map)
    else:
        df_to_npz(sdf, OUT_DIR / f"{name}.npz", label_map=None)

    print(f"{name}: {len(sdf)}")

summary = {
    "protocol": "pretrain_on_A__finetune_on_B__joint_retrain_on_AplusB__evaluate_unseen_locations_on_both_domains",
    "single_receiver_samples_only": True,
    "paired_AB_samples": False,
    "fused_AB_input": False,
    "seed": SEED,
    "A_val_ratio": A_VAL_RATIO,
    "B_val_ratio": B_VAL_RATIO,
    "target_eval_unseen_locations": sorted(list(target_unseen_loc_set)),
    "counts": {
        "all_windows": int(len(df)),
        "A_all": int(len(df_A)),
        "B_all": int(len(df_B)),
        "A_train": int(len(A_train)),
        "A_val": int(len(A_val)),
        "B_finetune": int(len(B_finetune)),
        "B_val": int(len(B_val)),
        "A_eval_unseen_locations": int(len(A_eval_unseen_locations)),
        "B_eval_unseen_locations": int(len(B_eval_unseen_locations)),
        "target_eval_unseen_locations_all": int(len(target_eval_unseen_locations_all)),
        "AB_joint_retrain": int(len(AB_joint_retrain)),
        "AB_joint_val": int(len(AB_joint_val)),
    },
    "num_labels": int(len(label_map)),
    "receiver_counts": {str(k): int(v) for k, v in df["receiver_domain"].value_counts().to_dict().items()},
    "label_counts_all": {str(k): int(v) for k, v in df["anchor_label"].value_counts().to_dict().items()},
    "joint_retrain_receiver_counts": {
        str(k): int(v) for k, v in AB_joint_retrain["receiver_domain"].value_counts().to_dict().items()
    },
    "joint_val_receiver_counts": {
        str(k): int(v) for k, v in AB_joint_val["receiver_domain"].value_counts().to_dict().items()
    },
}

with open(OUT_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved summary:", OUT_DIR / "dataset_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))


import gc
import tensorflow as tf
gc.collect()
tf.keras.backend.clear_session()
print("Session cleared.")

Found amp files: 9632
Total windows: 9632
receiver_domain
B    4887
A    4745
Name: count, dtype: int64
anchor_label
MiddleUp            594
Empty               594
LeftDown            594
Corner_LeftUp       592
Corner_RightDown    592
Corner_RightUp      591
RightUp             591
LeftUp              590
Corner_LeftDown     589
RightMid            586
LeftMid             570
MiddleDown          565
RightDown           525
LeftUp_Pred         317
Empty_Pred          300
Center              295
LeftDown_Far        295
LeftUp_Near         293
Wall                282
Corner              277
Name: count, dtype: int64
Saved manifest: /home/tonyliao/Location/dataset_build_hybrid/all_windows_manifest.csv
num_labels: 13
{'Corner_LeftDown': 0, 'Corner_LeftUp': 1, 'Corner_RightDown': 2, 'Corner_RightUp': 3, 'Empty': 4, 'LeftDown': 5, 'LeftMid': 6, 'LeftUp': 7, 'MiddleDown': 8, 'MiddleUp': 9, 'RightDown': 10, 'RightMid': 11, 'RightUp': 12}

A split label counts:
A_train:
anchor_label
Corner_Lef

2026-04-13 13:43:22.880824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 13:43:22.886486: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776087802.893846 1562183 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776087802.896084 1562183 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776087802.901679 1562183 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Session cleared.
